# SaaS Subscription Churn Prediction: Classification & Evaluation

**Project:** RavenStack SaaS dataset (500 accounts: 390 retained, 110 churned)
**Goal:** Predict which accounts are likely to churn, using account, subscription, usage and support data.

**Notebook flow**
1. Load data and define the prediction date
2. Build account-level features
3. Investigate missing values (MRR)
4. Preprocessing and train/validation/test split
5. Train and compare models
6. Cross-validation, feature importance, threshold selection
7. Final test evaluation and saving outputs
8. Data quality notes and limitations (end of notebook)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import joblib


## 1. Load Cleaned Data
Loads the five cleaned tables (accounts, subscriptions, feature usage, support tickets, churn events) plus `account_data.csv`, and converts all date columns to datetime.

In [2]:
from pathlib import Path

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
data_dir = project_dir / "data" / "cleaned"

accounts = pd.read_csv(data_dir / "ravenstack_accounts_clean.csv")
subscriptions = pd.read_csv(data_dir / "ravenstack_subscriptions_clean.csv")
feature_usage = pd.read_csv(data_dir / "ravenstack_feature_usage_clean.csv")
support_tickets = pd.read_csv(data_dir / "ravenstack_support_tickets_clean.csv")
churn_events = pd.read_csv(data_dir / "ravenstack_churn_events_clean.csv")
account_data = pd.read_csv(data_dir / "account_data.csv")

for df, columns in [
    (accounts, ["signup_date"]),
    (subscriptions, ["start_date", "end_date"]),
    (feature_usage, ["usage_date"]),
    (support_tickets, ["submitted_at", "closed_at"]),
    (churn_events, ["churn_date"]),
    (account_data, ["signup_date", "churn_date"])
]:
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")

print("Datasets loaded successfully.")
print("Accounts:", accounts.shape)
print("Account data:", account_data.shape)
print("Prediction date exists in account_data:", "prediction_date" in account_data.columns)

Datasets loaded successfully.
Accounts: (500, 10)
Account data: (500, 33)
Prediction date exists in account_data: False


## 2. Observation Period, Churn Date and Prediction Date
- **Observation end:** latest date found across all tables.
- **Churn date:** taken from `churn_events` for churned accounts.
- **Prediction date:** 30 days before churn for churned accounts, and the observation end for retained accounts.
- Dates that are missing or fall before signup are counted here so they can be investigated.

In [3]:
# Observation period end
observation_end = max(
    feature_usage["usage_date"].max(),
    support_tickets["submitted_at"].max(),
    subscriptions["end_date"].dropna().max(),
    churn_events["churn_date"].max()
)

# Churn dates from churn events
churned_accounts = accounts.loc[
    accounts["churn_flag"] == 1, "account_id"
]

account_churn_date = (
    churn_events[
        churn_events["account_id"].isin(churned_accounts)
    ]
    .groupby("account_id")["churn_date"]
    .min()
)

accounts["churn_date"] = accounts["account_id"].map(account_churn_date)

# Prediction cutoff: 30 days before churn, otherwise observation end
accounts["prediction_date"] = pd.Timestamp(observation_end)

churned_mask = accounts["churn_flag"].eq(1)

accounts.loc[churned_mask, "prediction_date"] = (
    accounts.loc[churned_mask, "churn_date"]
    - pd.Timedelta(days=30)
)

# Flag invalid dates
invalid_dates = (
    accounts["prediction_date"].isna()
    | (accounts["prediction_date"] < accounts["signup_date"])
)

print("Observation end:", observation_end)
print("Missing churn dates:", accounts.loc[churned_mask, "churn_date"].isna().sum())
print("Missing prediction dates:", accounts["prediction_date"].isna().sum())
print("Prediction dates before signup:", (
    accounts["prediction_date"] < accounts["signup_date"]
).sum())
print("Total invalid prediction dates:", invalid_dates.sum())

Observation end: 2024-12-31 00:00:00
Missing churn dates: 35
Missing prediction dates: 35
Prediction dates before signup: 16
Total invalid prediction dates: 51


## 3. Feature Engineering
One row per account, built from records up to the prediction date:
- **Subscription:** average MRR, upgrades, downgrades, auto-renew count
- **Usage:** total usage, duration, errors, beta feature usage
- **Support:** ticket count, resolution time, response time, satisfaction, escalations
- **Account:** industry, country, plan tier, seats, trial flag, tenure

In [4]:
# Basic customer information
model_data = accounts[[
    "account_id",
    "industry",
    "country",
    "plan_tier",
    "seats",
    "is_trial",
    "churn_flag",
    "signup_date",
    "prediction_date"
]].copy()

# Customer tenure at prediction date
model_data["tenure_months"] = (
    model_data["prediction_date"] -
    model_data["signup_date"]
).dt.days / 30.44


# --------------------------------------------------
# 1. Subscription information before prediction date
# --------------------------------------------------

subscription_pre = subscriptions.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

subscription_pre = subscription_pre[
    subscription_pre["start_date"]
    <= subscription_pre["prediction_date"]
]

subscription_summary = (
    subscription_pre
    .groupby("account_id")
    .agg(
        avg_mrr=("mrr_amount", "mean"),
        upgrade_count=("upgrade_flag", "sum"),
        downgrade_count=("downgrade_flag", "sum"),
        auto_renew_count=("auto_renew_flag", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 2. Feature usage before prediction date
# --------------------------------------------------

usage_pre = feature_usage.merge(
    subscriptions[["subscription_id", "account_id"]],
    on="subscription_id",
    how="left"
)

usage_pre = usage_pre.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

usage_pre = usage_pre[
    usage_pre["usage_date"]
    <= usage_pre["prediction_date"]
]

usage_summary = (
    usage_pre
    .groupby("account_id")
    .agg(
        total_usage=("usage_count", "sum"),
        total_usage_duration=("usage_duration_secs", "sum"),
        total_errors=("error_count", "sum"),
        beta_feature_usage=("is_beta_feature", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 3. Support information before prediction date
# --------------------------------------------------

support_pre = support_tickets.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

support_pre = support_pre[
    support_pre["submitted_at"]
    <= support_pre["prediction_date"]
]

support_summary = (
    support_pre
    .groupby("account_id")
    .agg(
        total_tickets=("ticket_id", "count"),
        avg_resolution_hours=("resolution_time_hours", "mean"),
        avg_response_minutes=("first_response_time_minutes", "mean"),
        avg_satisfaction=("satisfaction_score", "mean"),
        escalated_tickets=("escalation_flag", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 4. Combine all information
# --------------------------------------------------

model_data = (
    model_data
    .merge(
        subscription_summary,
        on="account_id",
        how="left"
    )
    .merge(
        usage_summary,
        on="account_id",
        how="left"
    )
    .merge(
        support_summary,
        on="account_id",
        how="left"
    )
)

print("Model dataset shape:", model_data.shape)
model_data.head()

Model dataset shape: (500, 23)


,account_id,industry,country,plan_tier,seats,is_trial,churn_flag,signup_date,prediction_date,tenure_months,...,auto_renew_count,total_usage,total_usage_duration,total_errors,beta_feature_usage,total_tickets,avg_resolution_hours,avg_response_minutes,avg_satisfaction,escalated_tickets
0,A-2e4581,EdTech,United States,Basic,9,0,0,2024-10-16,2024-12-31,2.496715,...,8.0,535.0,152339.0,38.0,4.0,2.0,23.000000,91.000000,3.000000,0.0
1,A-43a9e3,FinTech,India,Basic,18,0,1,2023-08-17,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A-0a282f,DevTools,United States,Basic,1,0,0,2024-08-27,2024-12-31,4.139290,...,15.0,821.0,251210.0,48.0,5.0,3.0,43.666667,63.666667,4.666667,0.0
3,A-1f0ac7,HealthTech,United Kingdom,Basic,24,1,0,2023-08-27,2024-12-31,16.162943,...,5.0,382.0,102528.0,21.0,5.0,2.0,29.000000,174.000000,NaN,0.0
4,A-ce550d,HealthTech,United States,Enterprise,35,0,1,2024-10-27,2024-11-28,1.051248,...,3.0,559.0,209127.0,28.0,4.0,7.0,42.285714,107.857143,3.800000,1.0


## 4. Missing MRR Investigation
Checks which accounts have no `avg_mrr`, whether they have a prediction date, and how this relates to churn status.

In [5]:
# Investigate Missing MRR
missing_mrr = model_data[model_data["avg_mrr"].isna()]

print("Missing MRR accounts:", len(missing_mrr))
print("\nAvailable columns:")
print(model_data.columns.tolist())

print("\nMissing MRR by churn status:")
print(missing_mrr["churn_flag"].value_counts())

print("\nPrediction dates for missing MRR:")
print(missing_mrr["prediction_date"].isna().value_counts())

print("\nChurned accounts without prediction dates:")
display(
    model_data.loc[
        (model_data["churn_flag"] == 1) &
        model_data["prediction_date"].isna()
    ].head(10)
)

print("\nMissing-MRR accounts with valid prediction dates:")
display(
    missing_mrr.loc[
        missing_mrr["prediction_date"].notna()
    ].head(15)
)

Missing MRR accounts: 62

Available columns:
['account_id', 'industry', 'country', 'plan_tier', 'seats', 'is_trial', 'churn_flag', 'signup_date', 'prediction_date', 'tenure_months', 'avg_mrr', 'upgrade_count', 'downgrade_count', 'auto_renew_count', 'total_usage', 'total_usage_duration', 'total_errors', 'beta_feature_usage', 'total_tickets', 'avg_resolution_hours', 'avg_response_minutes', 'avg_satisfaction', 'escalated_tickets']

Missing MRR by churn status:
churn_flag
1    62
Name: count, dtype: int64

Prediction dates for missing MRR:
prediction_date
True     35
False    27
Name: count, dtype: int64

Churned accounts without prediction dates:


,account_id,industry,country,plan_tier,seats,is_trial,churn_flag,signup_date,prediction_date,tenure_months,...,auto_renew_count,total_usage,total_usage_duration,total_errors,beta_feature_usage,total_tickets,avg_resolution_hours,avg_response_minutes,avg_satisfaction,escalated_tickets
1,A-43a9e3,FinTech,India,Basic,18,0,1,2023-08-17,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,A-10b8da,DevTools,United States,Enterprise,14,0,1,2023-05-08,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,A-c4ea60,FinTech,Germany,Enterprise,18,0,1,2023-03-09,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,A-35083d,HealthTech,India,Basic,19,0,1,2023-04-28,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42,A-7f8241,EdTech,United States,Pro,34,0,1,2023-03-14,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55,A-4c38bc,HealthTech,United Kingdom,Basic,36,0,1,2024-03-09,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69,A-5b4ebb,Cybersecurity,United Kingdom,Basic,21,1,1,2024-07-10,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,A-8b25f2,Cybersecurity,United States,Pro,6,0,1,2024-02-17,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,A-503d5a,FinTech,United States,Enterprise,4,0,1,2023-10-21,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117,A-95b24a,FinTech,United States,Pro,38,0,1,2023-06-06,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Missing-MRR accounts with valid prediction dates:


,account_id,industry,country,plan_tier,seats,is_trial,churn_flag,signup_date,prediction_date,tenure_months,...,auto_renew_count,total_usage,total_usage_duration,total_errors,beta_feature_usage,total_tickets,avg_resolution_hours,avg_response_minutes,avg_satisfaction,escalated_tickets
8,A-7dacce,Cybersecurity,Canada,Enterprise,12,0,1,2024-09-10,2024-08-19,-0.722733,...,NaN,374.0,116499.0,25.0,4.0,4.0,32.5,75.25,3.666667,0.0
10,A-592832,Cybersecurity,United States,Basic,18,0,1,2024-09-25,2024-09-04,-0.689882,...,NaN,861.0,259364.0,48.0,12.0,2.0,28.5,95.50,5.000000,0.0
22,A-4e44e8,DevTools,United States,Enterprise,3,0,1,2024-08-20,2024-07-26,-0.821288,...,NaN,528.0,171071.0,26.0,5.0,2.0,46.0,110.00,3.500000,0.0
32,A-08e34e,EdTech,United States,Basic,14,0,1,2024-06-27,2024-06-13,-0.459921,...,NaN,432.0,132516.0,22.0,5.0,2.0,30.5,92.50,3.000000,0.0
50,A-b2225d,FinTech,United States,Basic,47,0,1,2024-03-08,2024-03-01,-0.229961,...,NaN,369.0,124193.0,39.0,4.0,NaN,NaN,NaN,NaN,NaN
54,A-32728c,DevTools,United States,Pro,20,0,1,2023-01-03,2022-12-26,-0.262812,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,A-271f46,DevTools,United States,Pro,38,0,1,2023-02-09,2023-02-02,-0.229961,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105,A-f19409,DevTools,United States,Basic,23,0,1,2023-03-16,2023-02-24,-0.657030,...,NaN,11.0,2013.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
111,A-e83938,DevTools,United Kingdom,Pro,2,1,1,2023-11-21,2024-01-20,1.971091,...,NaN,128.0,37288.0,9.0,1.0,2.0,40.0,33.50,NaN,0.0
124,A-5a184f,DevTools,Germany,Enterprise,1,0,1,2023-06-19,2023-06-23,0.131406,...,NaN,153.0,45004.0,3.0,2.0,NaN,NaN,NaN,NaN,NaN


## 5. Target and Feature Selection
- **Target:** `churn_flag` (1 = churned, 0 = retained)
- **Features:** 19 columns (3 categorical, 16 numeric)

In [6]:
target = "churn_flag"

features = [
    "plan_tier",
    "industry",
    "country",
    "seats",
    "is_trial",
    "avg_mrr",
    "upgrade_count",
    "downgrade_count",
    "auto_renew_count",
    "total_usage",
    "total_usage_duration",
    "total_errors",
    "beta_feature_usage",
    "total_tickets",
    "avg_resolution_hours",
    "avg_response_minutes",
    "avg_satisfaction",
    "escalated_tickets",
    "tenure_months"
]

X = model_data[features]
y = model_data[target]

print("Number of features:", len(features))
print("Target distribution:")
print(y.value_counts())

Number of features: 19
Target distribution:
churn_flag
0    390
1    110
Name: count, dtype: int64


In [7]:
categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['plan_tier', 'industry', 'country']

Numerical features:
['seats', 'is_trial', 'avg_mrr', 'upgrade_count', 'downgrade_count', 'auto_renew_count', 'total_usage', 'total_usage_duration', 'total_errors', 'beta_feature_usage', 'total_tickets', 'avg_resolution_hours', 'avg_response_minutes', 'avg_satisfaction', 'escalated_tickets', 'tenure_months']


## 6. Preprocessing Pipeline
- **Numeric:** median imputation
- **Categorical:** most-frequent imputation + one-hot encoding

In [8]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numerical_features
    ),
    (
        "cat",
        categorical_transformer,
        categorical_features
    )
])

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## 7. Train / Validation / Test Split
Stratified 60 / 20 / 20 split (300 / 100 / 100 accounts), so the churn ratio stays the same in every part.

In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nValidation target distribution:")
print(y_val.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training samples: 300
Validation samples: 100
Testing samples: 100

Training target distribution:
churn_flag
0    234
1     66
Name: count, dtype: int64

Validation target distribution:
churn_flag
0    78
1    22
Name: count, dtype: int64

Testing target distribution:
churn_flag
0    78
1    22
Name: count, dtype: int64


In [10]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (300, 31)
Processed testing shape: (100, 31)


## 8. Model Training
Three models are trained on the training set:
1. Logistic Regression (balanced class weights)
2. Random Forest (balanced class weights)
3. Gradient Boosting

In [11]:
logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_train_processed,
    y_train
)

print("Balanced Logistic Regression trained.")

Balanced Logistic Regression trained.


c:\Users\Pratham\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [12]:
random_forest_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42
)

random_forest_model.fit(
    X_train_processed,
    y_train
)

print("Balanced Random Forest trained.")

Balanced Random Forest trained.


In [13]:
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=5,
    random_state=42
)

gradient_boosting_model.fit(
    X_train_processed,
    y_train
)

print("Gradient Boosting trained.")

Gradient Boosting trained.


## 9. Model Comparison
Models are compared on the validation set using Accuracy, Precision, Recall, F1 and ROC-AUC. Accuracy alone is not enough because the classes are imbalanced (78% retained), so interpret all metrics together and read the notes at the end of the notebook.

In [14]:
def evaluate_model(model, X_eval_processed, y_eval, threshold=0.5):
    probabilities = model.predict_proba(X_eval_processed)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y_eval, predictions),
        "Precision": precision_score(y_eval, predictions, zero_division=0),
        "Recall": recall_score(y_eval, predictions, zero_division=0),
        "F1": f1_score(y_eval, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, probabilities)
    }

In [15]:
model_results = pd.DataFrame({
    "Logistic Regression": evaluate_model(
        logistic_model, X_val_processed, y_val
    ),
    "Random Forest": evaluate_model(
        random_forest_model, X_val_processed, y_val
    ),
    "Gradient Boosting": evaluate_model(
        gradient_boosting_model, X_val_processed, y_val
    )
}).T

model_results.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC
Logistic Regression,0.71,0.370,0.455,0.408,0.686
Random Forest,0.85,0.818,0.409,0.545,0.810
Gradient Boosting,0.90,0.929,0.591,0.722,0.805


In [16]:
gb_predictions = gradient_boosting_model.predict(
    X_test_processed
)

print(
    classification_report(
        y_test,
        gb_predictions,
        target_names=["Retained", "Churned"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

    Retained       0.91      0.96      0.94        78
     Churned       0.83      0.68      0.75        22

    accuracy                           0.90       100
   macro avg       0.87      0.82      0.84       100
weighted avg       0.90      0.90      0.90       100



## 10. Cross-Validation
5-fold stratified cross-validation of the Gradient Boosting pipeline, to check that the score is not just an effect of one particular split.

In [17]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

gb_pipeline = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ])),
    ("model", GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2,
        min_samples_leaf=5,
        random_state=42
    ))
])

cv_results = cross_validate(
    gb_pipeline,
    X,
    y,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ]
)

print("Cross-validation results:\n")

for metric in [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]:
    scores = cv_results[f"test_{metric}"]

    print(
        f"{metric.upper():10}: "
        f"{scores.mean():.3f} ± {scores.std():.3f}"
    )

Cross-validation results:

ACCURACY  : 0.926 ± 0.015
PRECISION : 0.924 ± 0.051
RECALL    : 0.727 ± 0.076
F1        : 0.810 ± 0.043
ROC_AUC   : 0.916 ± 0.033


In [18]:
print("Validation-set model comparison:")
print(model_results.round(3))

print("\nGradient Boosting cross-validation:")
print(pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "CV Mean": [
        cv_results["test_accuracy"].mean(),
        cv_results["test_precision"].mean(),
        cv_results["test_recall"].mean(),
        cv_results["test_f1"].mean(),
        cv_results["test_roc_auc"].mean()
    ]
}).round(3))

Validation-set model comparison:
                     Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression      0.71      0.370   0.455  0.408    0.686
Random Forest            0.85      0.818   0.409  0.545    0.810
Gradient Boosting        0.90      0.929   0.591  0.722    0.805

Gradient Boosting cross-validation:
      Metric  CV Mean
0   Accuracy    0.926
1  Precision    0.924
2     Recall    0.727
3         F1    0.810
4    ROC-AUC    0.916


## 11. Final Model and Feature Importance
The selected Gradient Boosting pipeline is trained on the training set, then feature importance is extracted from it.

In [19]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2,
        min_samples_leaf=5,
        random_state=42
    ))
])

final_model.fit(X_train, y_train)

print("Final Gradient Boosting model trained on training data.")

Final Gradient Boosting model trained on training data.


In [20]:
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
model = final_model.named_steps["model"]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(15))

,feature,importance
2,num__avg_mrr,0.527884
5,num__auto_renew_count,0.151750
15,num__tenure_months,0.146204
7,num__total_usage_duration,0.043798
8,num__total_errors,0.026664
3,num__upgrade_count,0.022841
0,num__seats,0.020328
10,num__total_tickets,0.018757
12,num__avg_response_minutes,0.012127
6,num__total_usage,0.007676


### 11.1 MRR and Permutation-Importance Diagnostics
- MRR missingness compared against churn status
- Churn rate across MRR quartiles
- Permutation importance on the held-out test set (drop in ROC-AUC when a feature is shuffled)

In [21]:
# MRR and Feature Importance Diagnostics
from sklearn.inspection import permutation_importance

# 1. Check MRR missingness against the target
print("MRR Missingness by Churn Status")
print(pd.crosstab(
    model_data["churn_flag"],
    model_data["avg_mrr"].isna(),
    margins=True
))

# 2. Check MRR relationship with churn and other features
numeric_data = model_data[features].select_dtypes(include=np.number)

print("\nCorrelation of Numeric Features with MRR")
print(
    numeric_data.corr()["avg_mrr"]
    .sort_values(ascending=False)
    .round(3)
)

mrr_corr = model_data[["avg_mrr", "churn_flag"]].corr().iloc[0, 1]
print(f"\nMRR-Churn Correlation (available MRR): {mrr_corr:.3f}")

# 3. Compare churn rates across MRR groups
mrr_data = model_data.dropna(subset=["avg_mrr"]).copy()
mrr_data["mrr_group"] = pd.qcut(
    mrr_data["avg_mrr"], 4, duplicates="drop"
)

mrr_summary = mrr_data.groupby(
    "mrr_group", observed=True
).agg(
    accounts=("churn_flag", "size"),
    churned=("churn_flag", "sum"),
    churn_rate=("churn_flag", "mean")
).reset_index()

mrr_summary["churn_rate"] *= 100

print("\nChurn Rate by MRR Group")
display(mrr_summary.round(2))

# 4. Measure predictive contribution on held-out test data
perm = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=30,
    random_state=42,
    n_jobs=-1
)

permutation_df = pd.DataFrame({
    "feature": X_test.columns,
    "mean_auc_drop": perm.importances_mean,
    "std_auc_drop": perm.importances_std
}).sort_values("mean_auc_drop", ascending=False)

print("\nPermutation Importance — Held-Out Test Set")
display(permutation_df.round(4))

# 5. Save the diagnostic results
output_dir = project_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

mrr_summary.to_csv(
    output_dir / "mrr_churn_diagnostics.csv",
    index=False
)

permutation_df.to_csv(
    output_dir / "permutation_importance.csv",
    index=False
)

print("\nDiagnostic results saved successfully.")

MRR Missingness by Churn Status
avg_mrr     False  True  All
churn_flag                  
0             390     0  390
1              48    62  110
All           438    62  500

Correlation of Numeric Features with MRR
avg_mrr                 1.000
seats                   0.705
avg_resolution_hours    0.094
upgrade_count           0.053
total_errors            0.051
beta_feature_usage      0.039
total_usage_duration    0.029
total_usage             0.028
avg_response_minutes    0.022
is_trial                0.014
downgrade_count         0.011
auto_renew_count        0.006
total_tickets          -0.030
escalated_tickets      -0.036
avg_satisfaction       -0.041
tenure_months          -0.089
Name: avg_mrr, dtype: float64

MRR-Churn Correlation (available MRR): -0.023

Churn Rate by MRR Group


,mrr_group,accounts,churned,churn_rate
0,"(-0.001, 1227.838]",110,21,19.09
1,"(1227.838, 1892.821]",109,11,10.09
2,"(1892.821, 2832.567]",109,7,6.42
3,"(2832.567, 14534.0]",110,9,8.18



Permutation Importance — Held-Out Test Set


,feature,mean_auc_drop,std_auc_drop
8,auto_renew_count,0.1102,0.0208
5,avg_mrr,0.1061,0.0375
18,tenure_months,0.0273,0.0173
10,total_usage_duration,0.0174,0.0065
13,total_tickets,0.0093,0.0037
11,total_errors,0.0072,0.0041
16,avg_satisfaction,0.0058,0.0052
14,avg_resolution_hours,0.0031,0.0039
7,downgrade_count,0.0028,0.0050
15,avg_response_minutes,0.0021,0.0067



Diagnostic results saved successfully.


In [22]:
# Feature importance from the fitted Gradient Boosting pipeline
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
model = final_model.named_steps["model"]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("Top 15 features influencing churn prediction:")
display(feature_importance.head(15))

Top 15 features influencing churn prediction:


,feature,importance
2,num__avg_mrr,0.527884
5,num__auto_renew_count,0.151750
15,num__tenure_months,0.146204
7,num__total_usage_duration,0.043798
8,num__total_errors,0.026664
3,num__upgrade_count,0.022841
0,num__seats,0.020328
10,num__total_tickets,0.018757
12,num__avg_response_minutes,0.012127
6,num__total_usage,0.007676


### 11.2 Clean Feature Names and Top 10 Features
Feature names are made readable, and the top 10 features are selected for the dashboard.

In [23]:
feature_importance["feature"] = (
    feature_importance["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
    .str.replace("_", " ", regex=False)
)

feature_importance["feature"] = (
    feature_importance["feature"]
    .str.title()
)

feature_importance.head(15)

,feature,importance
2,Avg Mrr,0.527884
5,Auto Renew Count,0.151750
15,Tenure Months,0.146204
7,Total Usage Duration,0.043798
8,Total Errors,0.026664
3,Upgrade Count,0.022841
0,Seats,0.020328
10,Total Tickets,0.018757
12,Avg Response Minutes,0.012127
6,Total Usage,0.007676


In [24]:
top_features = feature_importance.head(10).copy()

print("Top 10 features influencing churn:\n")
print(top_features)

Top 10 features influencing churn:

                 feature  importance
2                Avg Mrr    0.527884
5       Auto Renew Count    0.151750
15         Tenure Months    0.146204
7   Total Usage Duration    0.043798
8           Total Errors    0.026664
3          Upgrade Count    0.022841
0                  Seats    0.020328
10         Total Tickets    0.018757
12  Avg Response Minutes    0.012127
6            Total Usage    0.007676


In [25]:
# top_features.to_csv(
#     "feature_importance.csv",
#     index=False
# )

print("Feature importance saved successfully.")

Feature importance saved successfully.


## 12. Decision Threshold Selection
Different thresholds are tried on the **validation set** and the one with the best F1 is chosen. The test set is not used for this choice.

In [26]:
gb_val_probabilities = final_model.predict_proba(X_val)[:, 1]

threshold_values = np.arange(0.20, 0.71, 0.05)

threshold_results = []

for threshold in threshold_values:
    val_predictions = (
        gb_val_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_val, val_predictions),
        "Precision": precision_score(
            y_val, val_predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, val_predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, val_predictions, zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)

display(threshold_results.round(3))

best_threshold = threshold_results.loc[
    threshold_results["F1"].idxmax(),
    "Threshold"
]

print("Best validation threshold:", best_threshold)

,Threshold,Accuracy,Precision,Recall,F1
0,0.20,0.85,0.640,0.727,0.681
1,0.25,0.86,0.682,0.682,0.682
2,0.30,0.89,0.789,0.682,0.732
3,0.35,0.90,0.875,0.636,0.737
4,0.40,0.91,0.933,0.636,0.757
5,0.45,0.91,0.933,0.636,0.757
6,0.50,0.90,0.929,0.591,0.722
7,0.55,0.88,0.917,0.500,0.647
8,0.60,0.88,0.917,0.500,0.647
9,0.65,0.88,0.917,0.500,0.647


Best validation threshold: 0.4


## 13. Final Test-Set Evaluation
The chosen threshold is applied once to the held-out test set (100 accounts).

In [27]:
gb_probabilities = final_model.predict_proba(X_test)[:, 1]

final_test_predictions = (
    gb_probabilities >= best_threshold
).astype(int)

final_metrics = {
    "Accuracy": accuracy_score(y_test, final_test_predictions),
    "Precision": precision_score(
        y_test, final_test_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, final_test_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_test, final_test_predictions, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(y_test, gb_probabilities)
}

print("Final test performance:\n")

for metric, value in final_metrics.items():
    print(f"{metric}: {value:.3f}")

Final test performance:

Accuracy: 0.900
Precision: 0.800
Recall: 0.727
F1: 0.762
ROC-AUC: 0.876


## 14. Saving the Model
Saves the model pipeline and the selected threshold for the Streamlit app.

In [ ]:
joblib.dump(final_model, "churn_model.pkl")
joblib.dump(best_threshold, "churn_threshold.pkl")

print("Complete model pipeline saved successfully.")
print("Selected threshold:", best_threshold)

Complete model pipeline saved successfully.
Selected threshold: 0.4


In [29]:
print("Selected validation threshold:", best_threshold)
print("Untouched test samples:", len(y_test))
print("Actual churned:", int(y_test.sum()))

print("\nFinal test confusion matrix:")
print(confusion_matrix(y_test, final_test_predictions))

print("\nTest ROC-AUC:")
print(roc_auc_score(y_test, gb_probabilities))

Selected validation threshold: 0.4
Untouched test samples: 100
Actual churned: 22

Final test confusion matrix:
[[74  4]
 [ 6 16]]

Test ROC-AUC:
0.8764568764568764


## 15. Verify the Saved Model
Loads the saved pipeline from disk and checks the library versions, so the model can be reloaded safely in the app.

In [30]:
saved_model = joblib.load("../models/churn_model.pkl")

print("Model type:", type(saved_model))
print("Pipeline steps:", saved_model.named_steps.keys())
# print("Saved threshold:", joblib.load("churn_threshold.pkl"))

Model type: <class 'sklearn.pipeline.Pipeline'>
Pipeline steps: dict_keys(['preprocessor', 'model'])


In [31]:
import sys
import sklearn
import joblib

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Python executable: c:\Users\Pratham\AppData\Local\Python\pythoncore-3.14-64\python.exe
Scikit-learn: 1.8.0
Joblib: 1.5.3


In [32]:
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

model_path = project_dir / "models" / "churn_model.pkl"
model = joblib.load(model_path)

print("Model loaded:", type(model))
print("Pipeline steps:", model.named_steps.keys())

Model loaded: <class 'sklearn.pipeline.Pipeline'>
Pipeline steps: dict_keys(['preprocessor', 'model'])


In [33]:

model_path = Path("../models/churn_model.pkl")

print("Scikit-learn:", sklearn.__version__)
print("Model exists:", model_path.exists())

model = joblib.load(model_path)
print("Model type:", type(model))

Scikit-learn: 1.8.0
Model exists: True
Model type: <class 'sklearn.pipeline.Pipeline'>


## 16. Final Evaluation Outputs
Re-computes the final metrics, classification report and confusion matrix, and saves them to `outputs/` for the dashboard.

In [34]:
variables = [
    "X_test", "y_test",
    "X_train", "y_train",
    "final_model", "best_model",
    "final_test_predictions", "y_pred",
    "y_test_proba", "y_pred_proba",
    "threshold", "best_threshold"
]

for name in variables:
    if name in globals():
        obj = globals()[name]
        print(
            f"{name}: {type(obj).__name__}",
            f"| shape={obj.shape}" if hasattr(obj, "shape") else ""
        )

X_test: DataFrame | shape=(100, 19)
y_test: Series | shape=(100,)
X_train: DataFrame | shape=(300, 19)
y_train: Series | shape=(300,)
final_model: Pipeline 
final_test_predictions: ndarray | shape=(100,)
threshold: float64 | shape=()
best_threshold: float64 | shape=()


In [35]:
# 1. Generate churn probabilities
test_probabilities = final_model.predict_proba(X_test)[:, 1]

# 2. Apply the selected threshold
threshold_value = float(best_threshold)
test_predictions = (test_probabilities >= threshold_value).astype(int)

# 3. Calculate evaluation metrics
test_metrics = {
    "Accuracy": accuracy_score(y_test, test_predictions),
    "Precision": precision_score(y_test, test_predictions, zero_division=0),
    "Recall": recall_score(y_test, test_predictions, zero_division=0),
    "F1": f1_score(y_test, test_predictions, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, test_probabilities)
}

print(f"Threshold: {threshold_value:.3f}")
print("\nTest-set performance:")
display(pd.DataFrame([test_metrics]).round(3))

print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    labels=[0, 1],
    target_names=["Retained", "Churned"],
    zero_division=0
))

# 4. Generate the confusion matrix
cm = confusion_matrix(y_test, test_predictions, labels=[0, 1])

cm_df = pd.DataFrame(
    cm,
    index=["Actual Retained", "Actual Churned"],
    columns=["Predicted Retained", "Predicted Churned"]
)

print("\nConfusion matrix:")
display(cm_df)

Threshold: 0.400

Test-set performance:


,Accuracy,Precision,Recall,F1,ROC-AUC
0,0.9,0.8,0.727,0.762,0.876



Classification report:
              precision    recall  f1-score   support

    Retained       0.93      0.95      0.94        78
     Churned       0.80      0.73      0.76        22

    accuracy                           0.90       100
   macro avg       0.86      0.84      0.85       100
weighted avg       0.90      0.90      0.90       100


Confusion matrix:


,Predicted Retained,Predicted Churned
Actual Retained,74,4
Actual Churned,6,16


In [36]:
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

output_dir = project_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Save confusion matrix
cm_df.to_csv(output_dir / "confusion_matrix.csv")

# Save final test metrics
pd.DataFrame([test_metrics]).to_csv(
    output_dir / "test_metrics.csv",
    index=False
)

print("Saved successfully:")
print(output_dir / "confusion_matrix.csv")
print(output_dir / "test_metrics.csv")

Saved successfully:
c:\Users\Pratham\OneDrive\Desktop\Saas_project_details\outputs\confusion_matrix.csv
c:\Users\Pratham\OneDrive\Desktop\Saas_project_details\outputs\test_metrics.csv


## 17. Final Data Diagnostics
Last check of the dataset: missing MRR, missing or invalid dates, and tenure.

In [39]:
print("Dataset shape:", model_data.shape)
print("Missing MRR:", model_data["avg_mrr"].isna().sum())

print("\nChurn by MRR availability:")
print(
    model_data.assign(
        mrr_status=model_data["avg_mrr"].isna().map(
            {True: "Missing", False: "Available"}
        )
    ).groupby("mrr_status")["churn_flag"].agg(
        accounts="count", churned="sum", churn_rate="mean"
    )
)

print("\nMissing MRR by prediction-date availability:")
print(
    model_data.loc[model_data["avg_mrr"].isna(), "prediction_date"]
    .isna().value_counts()
)

print("\nDate and tenure diagnostics:")
print("Missing signup dates:", model_data["signup_date"].isna().sum())
print("Missing prediction dates:", model_data["prediction_date"].isna().sum())
print("Negative tenure:", (model_data["tenure_months"] < 0).sum())
print(
    "Prediction before signup:",
    (model_data["prediction_date"] < model_data["signup_date"]).sum()
)

Dataset shape: (500, 23)
Missing MRR: 62

Churn by MRR availability:
            accounts  churned  churn_rate
mrr_status                               
Available        438       48    0.109589
Missing           62       62    1.000000

Missing MRR by prediction-date availability:
prediction_date
True     35
False    27
Name: count, dtype: int64

Date and tenure diagnostics:
Missing signup dates: 0
Missing prediction dates: 35
Negative tenure: 16
Prediction before signup: 16


## 18. Data Quality Notes & Limitations

**What the diagnostics in this notebook show**
- 62 accounts have no `avg_mrr`, and **all 62 are churned** (0 of 390 retained accounts). Missing MRR therefore carries information about the target.
- Median imputation turns that missingness into a distinct MRR value, so the model can learn "imputed MRR = churned". `avg_mrr` is also the top feature (about 53% importance).
- 35 churned accounts have no churn date, so no prediction date. The filter `start_date <= prediction_date` then removes all their subscriptions.
- 16 accounts have a prediction date before signup (churned within 30 days of signup), which gives negative tenure.
- The retained and churned accounts use different time windows. Retained accounts are measured at the observation end, and churned accounts 30 days before churn.
- Accuracy of 0.90 should be compared with the 78% that an "everyone retained" model would score.
- In cells 25 and 28 the save commands are commented out, even though the printed messages say "saved". The saved model file comes from an earlier run.

**What this means**
The scores in this notebook (accuracy about 0.90, ROC-AUC about 0.88) are partly produced by the missing-MRR pattern and the different time windows. They do not measure how well churn can be predicted for new accounts.

**What was done about it**
A corrected version was rebuilt with a fixed cutoff date per account, features limited to information available before that date, and no median-filled missingness signal. On this data it gives a ROC-AUC of about 0.50, which is no better than random.
The dataset is synthetic, and its churn label is almost independent of the other tables.

**Conclusion**
- This notebook is a record of the first modelling attempt and of the data problems found.
- The real contribution of the project is the data validation, the leakage diagnosis and the cleaned pipeline.
- Any reported model performance should be presented together with these limitations.